# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kaant7/flyrank-internship-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [20]:
import os
from dotenv import load_dotenv
import duckdb

load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")
assert HF_TOKEN is not None, "HF_TOKEN .env dosyasında bulunamadı"

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

dim_clients                     104 rows
dim_content                 519,606 rows
fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of analysis: One row = one content item (content_hash_id), aggregated over a fixed one-month window: 2026-03-01 to 2026-03-31.

Table(s): fact_content_daily_performance (daily grain: report_date + client_hash_id + content_hash_id), joined to dim_content for content metadata, filtered to month=2026-03.

Why this window: A mid-panel month avoids the sealed final month (fact_daily_sample, which is June 2026 — the natural outcome window for any past→future label) and gives enough days per content item to compute stable aggregates without touching the whole 79M-row table.

What I'd predict/rank: Same as Week 2 — is_declining, a proxy label derived from trend movement within this window, used to rank which observed signals associate with it.

What I deliberately exclude: Raw query-level data (fact_content_query_90d) for this notebook — it's a different grain (client × content × query hash) and would need its own join logic; I'm keeping this contract to the daily performance grain only.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

sample = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, gsc_impressions, gsc_clicks
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
    LIMIT 5
""").df()
sample

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature (model input):

gsc_impressions — visibility signal, known at decision time
gsc_clicks — click signal
gsc_avg_position — ranking signal
ga4_sessions — traffic/engagement signal
scroll_events — engagement depth signal

Label / proxy (target):

is_declining — a proxy label I'll derive from the gsc_impressions trend within March: comparing the first half of the month to the second half. There's no ready-made trend column in the warehouse (unlike the starter dataset's trend_direction), so I'm building this myself from the daily fact table.

Context (used for joins/grouping, not as a feature):

client_hash_id, content_hash_id — join keys and grouping only; the codes themselves carry no meaning and will never be model features
report_date — defines the time window, not a direct feature
client_has_gsc, client_has_ga4, gsc_data_available, ga4_data_available — availability flags, used for filtering, not as model inputs

Excluded (deliberately left out):

sessions_paid, sessions_social, sessions_referral, sessions_direct — Lane 1's question is about organic search signals (visibility/clicks/engagement/movement in a search context); paid and social traffic follow a different dynamic and are out of scope here
sessions_ai, ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other — AI-referral signals are known to be sparse (roughly 30K rows out of 79M in the full release), so including them would add noise rather than signal to this analysis
ga4_pageviews, ga4_users, ga4_engaged_sessions, ga4_total_engagement_sec — ga4_sessions and scroll_events already give a reasonable engagement signal; I'm excluding the rest to stay within the 5-feature limit for this notebook

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

field_types = {
    "feature": ["gsc_impressions", "gsc_clicks", "gsc_avg_position", "ga4_sessions", "scroll_events"],
    "label_proxy": ["is_declining (derived from gsc_impressions trend within March)"],
    "context": ["client_hash_id", "content_hash_id", "report_date",
                "client_has_gsc", "client_has_ga4", "gsc_data_available", "ga4_data_available"],
    "excluded": ["sessions_paid", "sessions_social", "sessions_referral", "sessions_direct",
                 "sessions_ai", "ai_chatgpt", "ai_perplexity", "ai_gemini", "ai_copilot", "ai_claude", "ai_meta", "ai_other",
                 "ga4_pageviews", "ga4_users", "ga4_engaged_sessions", "ga4_total_engagement_sec"],
}

for bucket, fields in field_types.items():
    print(f"\n{bucket.upper()} ({len(fields)}):")
    for f in fields:
        print(" -", f)


FEATURE (5):
 - gsc_impressions
 - gsc_clicks
 - gsc_avg_position
 - ga4_sessions
 - scroll_events

LABEL_PROXY (1):
 - is_declining (derived from gsc_impressions trend within March)

CONTEXT (7):
 - client_hash_id
 - content_hash_id
 - report_date
 - client_has_gsc
 - client_has_ga4
 - gsc_data_available
 - ga4_data_available

EXCLUDED (16):
 - sessions_paid
 - sessions_social
 - sessions_referral
 - sessions_direct
 - sessions_ai
 - ai_chatgpt
 - ai_perplexity
 - ai_gemini
 - ai_copilot
 - ai_claude
 - ai_meta
 - ai_other
 - ga4_pageviews
 - ga4_users
 - ga4_engaged_sessions
 - ga4_total_engagement_sec


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three queries below prove the claims made in Sections 1–2: that the grain is one row per content item per day per client, that the March slice has the row count and date span I expect, and how many rows actually have usable GSC data.

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

grain_check = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS unique_combinations
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
""").df()
grain_check

,total_rows,unique_combinations
0,9841378,9841378


In [24]:
span_check = con.sql(f"""
    SELECT COUNT(*) AS row_count,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date,
           COUNT(DISTINCT content_hash_id) AS unique_content_items,
           COUNT(DISTINCT client_hash_id) AS unique_clients
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
""").df()
span_check

,row_count,min_date,max_date,unique_content_items,unique_clients
0,9841378,2026-03-01,2026-03-31,331437,55


In [25]:
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS rows_with_gsc,
        ROUND(100.0 * COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) / COUNT(*), 1) AS pct_with_gsc
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
""").df()
availability_check

,total_rows,rows_with_gsc,pct_with_gsc
0,9841378,3611061,36.7


Verified: Grain confirmed (no duplicate combinations). March 2026 slice: 9,841,378 rows, 331,437 unique content items, 55 clients, spanning 2026-03-01 to 2026-03-31 exactly. Only 36.7% of rows (3,611,061) have gsc_data_available IS TRUE — meaning nearly two-thirds of this month's rows have no usable GSC signal, likely because tracking hadn't started yet for some clients (per dim_clients.gsc_data_start). This directly shapes my feature build in Section 3: I'll filter to gsc_data_available IS TRUE before aggregating, otherwise I'd be treating "not tracked yet" as "zero traffic."

In [26]:
features = con.sql(f"""
    SELECT
        content_hash_id,
        SUM(gsc_impressions)                       AS total_impressions,
        SUM(gsc_clicks)                             AS total_clicks,
        AVG(gsc_avg_position)                       AS avg_position,
        SUM(ga4_sessions)                            AS total_sessions,
        SUM(scroll_events)                           AS total_scroll_events
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
    GROUP BY content_hash_id
""").df()

print("Shape:", features.shape)
features.head()

Shape: (176738, 6)


,content_hash_id,total_impressions,total_clicks,avg_position,total_sessions,total_scroll_events
0,content_952ce695b3f329e4,5154.0,4.0,8.120965,3.0,0.0
1,content_9f6957faecbbad11,196.0,0.0,61.178404,0.0,0.0
2,content_31646238db2e69ac,151.0,0.0,50.741694,1.0,0.0
3,content_c9559373721a6de4,96.0,0.0,8.634975,0.0,0.0
4,content_debb63e36c727731,15.0,0.0,5.850000,0.0,0.0


total_impressions — knowable at the decision moment because it's a sum of past daily GSC impressions within the observed window, no future data involved.

total_clicks — same: aggregated from past daily observations only.

avg_position — average of daily positions already recorded within the window.

total_sessions — aggregated GA4 sessions from the same past window.

total_scroll_events — aggregated engagement signal from the same past window.

In [27]:
trend = con.sql(f"""
    WITH daily AS (
        SELECT content_hash_id,
               CASE WHEN report_date <= '2026-03-15' THEN 'first_half' ELSE 'second_half' END AS half,
               gsc_impressions
        FROM {TABLES['fact_daily']}
        WHERE month = '2026-03' AND gsc_data_available IS TRUE
    )
    SELECT content_hash_id,
           SUM(CASE WHEN half = 'first_half' THEN gsc_impressions ELSE 0 END) AS imp_first_half,
           SUM(CASE WHEN half = 'second_half' THEN gsc_impressions ELSE 0 END) AS imp_second_half
    FROM daily
    GROUP BY content_hash_id
""").df()

trend["is_declining"] = (trend["imp_second_half"] < trend["imp_first_half"]).astype(int)
trend[["content_hash_id", "is_declining"]].head()

,content_hash_id,is_declining
0,content_952ce695b3f329e4,0
1,content_9f6957faecbbad11,0
2,content_31646238db2e69ac,0
3,content_c9559373721a6de4,1
4,content_debb63e36c727731,1


In [28]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

data = features.merge(trend[["content_hash_id", "is_declining", "imp_second_half"]], on="content_hash_id")
data = data.dropna()

X = data[["total_impressions", "total_clicks", "avg_position", "total_sessions", "total_scroll_events"]]
y = data["is_declining"]

model = LogisticRegression(max_iter=1000)
model.fit(X, y)
pred = model.predict_proba(X)[:, 1]

print("ROC AUC (honest, before leak):", roc_auc_score(y, pred))

ROC AUC (honest, before leak): 0.5569558504400418


In [29]:
# Deliberate leak: this column is literally derived FROM the label
data["leaked_feature"] = data["imp_second_half"]

X_leak = data[["total_impressions", "total_clicks", "avg_position", "total_sessions", "total_scroll_events", "leaked_feature"]]

model_leak = LogisticRegression(max_iter=1000)
model_leak.fit(X_leak, y)
pred_leak = model_leak.predict_proba(X_leak)[:, 1]

print("ROC AUC (WITH leak):", roc_auc_score(y, pred_leak))

ROC AUC (WITH leak): 1.0


In [30]:
# Delete the leaked column — keep only the honest feature set
data = data.drop(columns=["leaked_feature"])
print("Final honest ROC AUC (kept):", roc_auc_score(y, pred))

Final honest ROC AUC (kept): 0.5569558504400418


The leakage trap: I deliberately added leaked_feature = second-half March impressions — literally the same data used to compute is_declining. ROC AUC jumped from 0.557 (honest) to 1.0 (leaked) — a textbook circular result. The model wasn't learning anything; it was just reading the label back through a disguised column. I removed the leaked feature and kept the honest 0.557 as my real baseline.

What 0.557 tells me: This is barely above random guessing (0.5), meaning these five signals alone don't separate "declining" from "not declining" very well within a single month's first-half/second-half split. This is a weaker proxy label than the starter dataset's trend_direction (which reached 0.627–0.750 with the baseline/models) — likely because a 15-day vs. 15-day split within one month is noisier than a proper before/after window. This is useful evidence for later weeks: I may need a stronger label definition (e.g. a longer window, or minimum volume filtering) before modeling seriously.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This slice cannot tell me:

Whether tracking was simply missing, not zero traffic. Only 36.7% of March 2026 rows have gsc_data_available IS TRUE. The remaining 63.3% aren't necessarily pages with no visibility — they may be rows from before a client's GSC tracking started (dim_clients.gsc_data_start). Treating them as "zero" instead of "unknown" would silently bias any aggregate I compute.

Whether a real decline happened, versus noise from a short window. My is_declining label compares the first 15 days of March to the last 15 days — a very short split. A weak, noisy signal (ROC AUC 0.557, barely above random) suggests this window is too short and too close to the boundary between "real movement" and "normal day-to-day wiggle" to trust as a strong label.

Whether a drop is a real decline, consolidation, seasonality, or SERP-level click loss. The lane guide's Section 7 lists these look-alikes explicitly — a page can lose impressions because a sibling page absorbed the demand, because of a seasonal dip, or because the search results page itself changed, not because the content got worse. This slice, aggregated at the content-item level with no sibling-page or category grouping, cannot distinguish between these causes.

History depth is unbalanced across clients. The panel has 55 clients in this March slice, but each client's tracking started at a different date (an unbalanced panel, per the lane guide). A client with only a few weeks of history contributes noisier aggregates than one with a full year — this slice alone doesn't show me which is which without joining dim_clients.

Causation. Even a strong, clean signal here would only be a correlation — this data cannot prove that a signal caused visibility to change, only that they moved together within the observed window.

In [31]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.